In [1]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [5]:
import pandas as pd

df = pd.read_csv("all_product_review_summaries.csv")

In [6]:
df.head()

,product_id,product_title,product_category_leaf_id,product_category_top_id,product_category_breadcrumb,product_brand,product_model_number,product_price_currency,product_price_min,product_price_max,...,review_verified_purchase,review_repeat_buyer,review_helpful_votes,review_total_votes,review_price_at_review,review_currency,review_variant,review_images_count,data_source,scraped_at
0,1600459219006,2026 High Quality Custom Logo Polarized Eyewea...,33902,36,"Jewelry, Eyewear, Watches & Accessories > Eyew...",QMOON,QMZZ204,BDT,912.66,1058.19,...,True,False,0,NaN,NaN,NaN,NaN,4.0,alibaba.com,2026-08-07T05:29:17+06:00
1,1600459219006,2026 High Quality Custom Logo Polarized Eyewea...,33902,36,"Jewelry, Eyewear, Watches & Accessories > Eyew...",QMOON,QMZZ204,BDT,912.66,1058.19,...,True,True,1,NaN,NaN,NaN,NaN,7.0,alibaba.com,2026-08-07T05:29:17+06:00
2,1600459219006,2026 High Quality Custom Logo Polarized Eyewea...,33902,36,"Jewelry, Eyewear, Watches & Accessories > Eyew...",QMOON,QMZZ204,BDT,912.66,1058.19,...,True,False,0,NaN,NaN,NaN,NaN,3.0,alibaba.com,2026-08-07T05:29:17+06:00
3,1600459219006,2026 High Quality Custom Logo Polarized Eyewea...,33902,36,"Jewelry, Eyewear, Watches & Accessories > Eyew...",QMOON,QMZZ204,BDT,912.66,1058.19,...,True,False,0,NaN,NaN,NaN,NaN,4.0,alibaba.com,2026-08-07T05:29:17+06:00
4,1600459219006,2026 High Quality Custom Logo Polarized Eyewea...,33902,36,"Jewelry, Eyewear, Watches & Accessories > Eyew...",QMOON,QMZZ204,BDT,912.66,1058.19,...,True,True,0,NaN,NaN,NaN,NaN,4.0,alibaba.com,2026-08-07T05:29:17+06:00


In [7]:
print(df.shape)
print(df.columns)

(55056, 42)
Index(['product_id', 'product_title', 'product_category_leaf_id',
       'product_category_top_id', 'product_category_breadcrumb',
       'product_brand', 'product_model_number', 'product_price_currency',
       'product_price_min', 'product_price_max', 'product_moq', 'product_unit',
       'product_sales_volume_text', 'product_sku_ids',
       'product_default_sku_id', 'seller_company_id', 'seller_company_name',
       'seller_business_type', 'seller_register_country',
       'seller_years_on_platform', 'seller_average_rating',
       'seller_total_reviewed_orders', 'seller_trade_assurance_enabled',
       'seller_on_time_delivery_rate', 'review_source', 'review_id',
       'reviewer_name', 'reviewer_country', 'review_rating', 'review_title',
       'review_text', 'review_date', 'review_verified_purchase',
       'review_repeat_buyer', 'review_helpful_votes', 'review_total_votes',
       'review_price_at_review', 'review_currency', 'review_variant',
       'review_images_c

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55056 entries, 0 to 55055
Data columns (total 42 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   product_id                      55056 non-null  int64  
 1   product_title                   55056 non-null  object 
 2   product_category_leaf_id        55056 non-null  int64  
 3   product_category_top_id         55056 non-null  int64  
 4   product_category_breadcrumb     55056 non-null  object 
 5   product_brand                   32836 non-null  object 
 6   product_model_number            29565 non-null  object 
 7   product_price_currency          42726 non-null  object 
 8   product_price_min               42726 non-null  float64
 9   product_price_max               42726 non-null  float64
 10  product_moq                     55056 non-null  int64  
 11  product_unit                    55056 non-null  object 
 12  product_sales_volume_text       

In [9]:
df.isnull().sum()

,0
product_id,0
product_title,0
product_category_leaf_id,0
product_category_top_id,0
product_category_breadcrumb,0
product_brand,22220
product_model_number,25491
product_price_currency,12330
product_price_min,12330
product_price_max,12330


In [10]:
df.head(10)

,product_id,product_title,product_category_leaf_id,product_category_top_id,product_category_breadcrumb,product_brand,product_model_number,product_price_currency,product_price_min,product_price_max,...,review_verified_purchase,review_repeat_buyer,review_helpful_votes,review_total_votes,review_price_at_review,review_currency,review_variant,review_images_count,data_source,scraped_at
0,1600459219006,2026 High Quality Custom Logo Polarized Eyewea...,33902,36,"Jewelry, Eyewear, Watches & Accessories > Eyew...",QMOON,QMZZ204,BDT,912.66,1058.19,...,True,False,0,NaN,NaN,NaN,NaN,4.0,alibaba.com,2026-08-07T05:29:17+06:00
1,1600459219006,2026 High Quality Custom Logo Polarized Eyewea...,33902,36,"Jewelry, Eyewear, Watches & Accessories > Eyew...",QMOON,QMZZ204,BDT,912.66,1058.19,...,True,True,1,NaN,NaN,NaN,NaN,7.0,alibaba.com,2026-08-07T05:29:17+06:00
2,1600459219006,2026 High Quality Custom Logo Polarized Eyewea...,33902,36,"Jewelry, Eyewear, Watches & Accessories > Eyew...",QMOON,QMZZ204,BDT,912.66,1058.19,...,True,False,0,NaN,NaN,NaN,NaN,3.0,alibaba.com,2026-08-07T05:29:17+06:00
3,1600459219006,2026 High Quality Custom Logo Polarized Eyewea...,33902,36,"Jewelry, Eyewear, Watches & Accessories > Eyew...",QMOON,QMZZ204,BDT,912.66,1058.19,...,True,False,0,NaN,NaN,NaN,NaN,4.0,alibaba.com,2026-08-07T05:29:17+06:00
4,1600459219006,2026 High Quality Custom Logo Polarized Eyewea...,33902,36,"Jewelry, Eyewear, Watches & Accessories > Eyew...",QMOON,QMZZ204,BDT,912.66,1058.19,...,True,True,0,NaN,NaN,NaN,NaN,4.0,alibaba.com,2026-08-07T05:29:17+06:00
5,1600459219006,2026 High Quality Custom Logo Polarized Eyewea...,33902,36,"Jewelry, Eyewear, Watches & Accessories > Eyew...",QMOON,QMZZ204,BDT,912.66,1058.19,...,True,True,0,NaN,NaN,NaN,NaN,4.0,alibaba.com,2026-08-07T05:29:17+06:00
6,1600459219006,2026 High Quality Custom Logo Polarized Eyewea...,33902,36,"Jewelry, Eyewear, Watches & Accessories > Eyew...",QMOON,QMZZ204,BDT,912.66,1058.19,...,True,True,0,NaN,NaN,NaN,NaN,4.0,alibaba.com,2026-08-07T05:29:17+06:00
7,1600459219006,2026 High Quality Custom Logo Polarized Eyewea...,33902,36,"Jewelry, Eyewear, Watches & Accessories > Eyew...",QMOON,QMZZ204,BDT,912.66,1058.19,...,True,True,1,NaN,NaN,NaN,NaN,3.0,alibaba.com,2026-08-07T05:29:17+06:00
8,1600459219006,2026 High Quality Custom Logo Polarized Eyewea...,33902,36,"Jewelry, Eyewear, Watches & Accessories > Eyew...",QMOON,QMZZ204,BDT,912.66,1058.19,...,True,True,0,NaN,NaN,NaN,NaN,3.0,alibaba.com,2026-08-07T05:29:17+06:00
9,1600459219006,2026 High Quality Custom Logo Polarized Eyewea...,33902,36,"Jewelry, Eyewear, Watches & Accessories > Eyew...",QMOON,QMZZ204,BDT,912.66,1058.19,...,True,False,0,NaN,NaN,NaN,NaN,3.0,alibaba.com,2026-08-07T05:29:17+06:00


In [12]:
import pandas as pd

df = pd.read_csv("/content/all_product_review_summaries.csv")

print("Dataset shape:", df.shape)
print(df.columns.tolist())

Dataset shape: (55056, 42)
['product_id', 'product_title', 'product_category_leaf_id', 'product_category_top_id', 'product_category_breadcrumb', 'product_brand', 'product_model_number', 'product_price_currency', 'product_price_min', 'product_price_max', 'product_moq', 'product_unit', 'product_sales_volume_text', 'product_sku_ids', 'product_default_sku_id', 'seller_company_id', 'seller_company_name', 'seller_business_type', 'seller_register_country', 'seller_years_on_platform', 'seller_average_rating', 'seller_total_reviewed_orders', 'seller_trade_assurance_enabled', 'seller_on_time_delivery_rate', 'review_source', 'review_id', 'reviewer_name', 'reviewer_country', 'review_rating', 'review_title', 'review_text', 'review_date', 'review_verified_purchase', 'review_repeat_buyer', 'review_helpful_votes', 'review_total_votes', 'review_price_at_review', 'review_currency', 'review_variant', 'review_images_count', 'data_source', 'scraped_at']


In [14]:
df = df[[
    "review_text",
    "review_rating"
]]

In [15]:
print(df.head())

                                         review_text  review_rating
0  10/10 quality and feel really heavy and fit co...            5.0
1  Yesterday i received my sunglasses parcel, it’...            5.0
2  The product is nice and shipping is very fast....            5.0
3  Glasses are very distinctive and look great wh...            5.0
4  Product is 100% authentic. It meets my require...            5.0


In [16]:
df = df.dropna(subset=["review_text", "review_rating"])

In [17]:
df = df.drop_duplicates(subset=["review_text"])

In [18]:
print("Remaining reviews:", len(df))

Remaining reviews: 39389


In [19]:
print(df["review_rating"].value_counts().sort_index())

review_rating
5.0    39389
Name: count, dtype: int64


In [20]:
def create_sentiment(rating):
    if rating <= 2:
        return 0
    elif rating == 3:
        return 1
    else:
        return 2

df["label"] = df["review_rating"].apply(create_sentiment)

In [21]:
print(df["label"].value_counts().sort_index())

label
2    39389
Name: count, dtype: int64


In [22]:
print(df.head())

                                         review_text  review_rating  label
0  10/10 quality and feel really heavy and fit co...            5.0      2
1  Yesterday i received my sunglasses parcel, it’...            5.0      2
2  The product is nice and shipping is very fast....            5.0      2
3  Glasses are very distinctive and look great wh...            5.0      2
4  Product is 100% authentic. It meets my require...            5.0      2


In [23]:
df = df.rename(columns={
    "review_text": "text"
})

In [24]:
df = df[["text", "label"]]

In [25]:
print(df.head())
print(df.shape)

                                                text  label
0  10/10 quality and feel really heavy and fit co...      2
1  Yesterday i received my sunglasses parcel, it’...      2
2  The product is nice and shipping is very fast....      2
3  Glasses are very distinctive and look great wh...      2
4  Product is 100% authentic. It meets my require...      2
(39389, 2)


now We should remove the supplier's reply.

Otherwise BERT will learn from the seller's response as well as the customer's review, which can introduce noise/data leakage.

In [26]:
def clean_review(text):
    text = str(text)

    if "|Supplier's reply:|" in text:
        text = text.split("|Supplier's reply:|")[0]

    if "|...Show all|Supplier's reply:|" in text:
        text = text.split("|...Show all|Supplier's reply:|")[0]

    return text.strip()

In [27]:
df["text"] = df["text"].apply(clean_review)

In [28]:
df = df[df["text"].str.len() > 5]

Final dataset check

In [29]:
print(df.shape)
print(df.head(10))

(38432, 2)
                                                text  label
0  10/10 quality and feel really heavy and fit co...      2
1  Yesterday i received my sunglasses parcel, it’...      2
2  The product is nice and shipping is very fast....      2
3  Glasses are very distinctive and look great wh...      2
4  Product is 100% authentic. It meets my require...      2
5  Is a very good experience with this company an...      2
6  Well … positive words for alibaba and for us ,...      2
7  The glasses are high quality and look much bet...      2
8  Very happy with the product. Sunshine is very ...      2
9    Professional communication and on time response      2


In [30]:
print(df["label"].value_counts())

label
2    38432
Name: count, dtype: int64


In [31]:
print(df["label"].value_counts(normalize=True))

label
2    1.0
Name: proportion, dtype: float64


Split dataset

In [32]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

Training: 26902
Validation: 5765
Testing: 5765


BERT install

In [33]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

In [34]:
from transformers import BertTokenizer

model_name = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [35]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [36]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

In [37]:
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/26902 [00:00<?, ? examples/s]

Map:   0%|          | 0/5765 [00:00<?, ? examples/s]

Map:   0%|          | 0/5765 [00:00<?, ? examples/s]

In [38]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fine tuning BERT

In [39]:
print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 26902
})
Dataset({
    features: ['text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 5765
})
Dataset({
    features: ['text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 5765
})


In [40]:
train_dataset = train_dataset.remove_columns(
    [col for col in train_dataset.column_names
     if col not in ["input_ids", "token_type_ids", "attention_mask", "label"]]
)

val_dataset = val_dataset.remove_columns(
    [col for col in val_dataset.column_names
     if col not in ["input_ids", "token_type_ids", "attention_mask", "label"]]
)

test_dataset = test_dataset.remove_columns(
    [col for col in test_dataset.column_names
     if col not in ["input_ids", "token_type_ids", "attention_mask", "label"]]
)

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

In [41]:
print(train_dataset)

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 26902
})


In [42]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_fscore_support

def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = predictions.argmax(axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [43]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_results",

    num_train_epochs=3,

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    logging_steps=100,

    report_to="none"
)

In [44]:
from transformers import Trainer

trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)

Train BERT

In [45]:
trainer.train()

ImportError: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)

In [1]:
!pip install -U -q torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5

In [1]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())

PyTorch: 2.13.0+cu130
Torchvision: 0.28.0+cu130
CUDA: True


In [2]:
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)

ModuleNotFoundError: Could not import module 'BertForSequenceClassification'. Are this object's requirements defined correctly?

In [3]:
!pip uninstall -y torchvision torch torchaudio
!pip install -q torch torchvision torchaudio

Found existing installation: torchvision 0.28.0
Uninstalling torchvision-0.28.0:
  Successfully uninstalled torchvision-0.28.0
Found existing installation: torch 2.13.0
Uninstalling torch-2.13.0:
  Successfully uninstalled torch-2.13.0
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.9 MB/s eta 0:00:00


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.13.0+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
from transformers import BertTokenizer, BertForSequenceClassification

print("Transformers loaded successfully!")

Transformers loaded successfully!


In [3]:
model_name = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(model_name)

model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

print("BERT loaded successfully!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT loaded successfully!


In [4]:
import pandas as pd

df = pd.read_csv("/content/all_product_review_summaries.csv")

print(df.shape)
print(df.columns.tolist())

(55056, 42)
['product_id', 'product_title', 'product_category_leaf_id', 'product_category_top_id', 'product_category_breadcrumb', 'product_brand', 'product_model_number', 'product_price_currency', 'product_price_min', 'product_price_max', 'product_moq', 'product_unit', 'product_sales_volume_text', 'product_sku_ids', 'product_default_sku_id', 'seller_company_id', 'seller_company_name', 'seller_business_type', 'seller_register_country', 'seller_years_on_platform', 'seller_average_rating', 'seller_total_reviewed_orders', 'seller_trade_assurance_enabled', 'seller_on_time_delivery_rate', 'review_source', 'review_id', 'reviewer_name', 'reviewer_country', 'review_rating', 'review_title', 'review_text', 'review_date', 'review_verified_purchase', 'review_repeat_buyer', 'review_helpful_votes', 'review_total_votes', 'review_price_at_review', 'review_currency', 'review_variant', 'review_images_count', 'data_source', 'scraped_at']


In [5]:
df = df[["review_text", "review_rating"]].copy()

df = df.dropna(subset=["review_text", "review_rating"])

df = df.drop_duplicates(subset=["review_text"])

In [6]:
def clean_review(text):
    text = str(text)

    if "|Supplier's reply:|" in text:
        text = text.split("|Supplier's reply:|")[0]

    if "|...Show all|Supplier's reply:|" in text:
        text = text.split("|...Show all|Supplier's reply:|")[0]

    return text.strip()

df["review_text"] = df["review_text"].apply(clean_review)

df = df[df["review_text"].str.len() > 5]

df = df.reset_index(drop=True)

print("Total reviews:", len(df))

Total reviews: 38432


In [8]:
def create_sentiment(rating):
    if rating <= 2:
        return 0
    elif rating == 3:
        return 1
    else:
        return 2

df["label"] = df["review_rating"].apply(create_sentiment)

In [9]:
print(df["label"].value_counts().sort_index())

label
2    38432
Name: count, dtype: int64


In [10]:
print(df.head())

                                         review_text  review_rating  label
0  10/10 quality and feel really heavy and fit co...            5.0      2
1  Yesterday i received my sunglasses parcel, it’...            5.0      2
2  The product is nice and shipping is very fast....            5.0      2
3  Glasses are very distinctive and look great wh...            5.0      2
4  Product is 100% authentic. It meets my require...            5.0      2


In [11]:
df = df.rename(columns={
    "review_text": "text"
})

df = df[["text", "label"]]

print(df.head())

                                                text  label
0  10/10 quality and feel really heavy and fit co...      2
1  Yesterday i received my sunglasses parcel, it’...      2
2  The product is nice and shipping is very fast....      2
3  Glasses are very distinctive and look great wh...      2
4  Product is 100% authentic. It meets my require...      2


In [12]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

Training: 26902
Validation: 5765
Testing: 5765


In [13]:
from transformers import BertTokenizer

model_name = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(model_name)

In [14]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [15]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

In [16]:
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/26902 [00:00<?, ? examples/s]

Map:   0%|          | 0/5765 [00:00<?, ? examples/s]

Map:   0%|          | 0/5765 [00:00<?, ? examples/s]

In [17]:
keep_columns = [
    "input_ids",
    "token_type_ids",
    "attention_mask",
    "label"
]

train_dataset = train_dataset.select_columns(keep_columns)
val_dataset = val_dataset.select_columns(keep_columns)
test_dataset = test_dataset.select_columns(keep_columns)

In [18]:
print(train_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'label'],
    num_rows: 26902
})


In [19]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_fscore_support

def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = predictions.argmax(axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [21]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_results",

    num_train_epochs=3,

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    logging_steps=100,

    report_to="none"
)

In [22]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [23]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000029,0.000020,1.000000,1.000000,1.000000,1.000000
2,0.000011,0.000008,1.000000,1.000000,1.000000,1.000000
3,0.000008,0.000006,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=5046, training_loss=0.0030539129194229533, metrics={'train_runtime': 1911.5266, 'train_samples_per_second': 42.221, 'train_steps_per_second': 2.64, 'total_flos': 5308707872788992.0, 'train_loss': 0.0030539129194229533, 'epoch': 3.0})

In [24]:
test_results = trainer.evaluate(test_dataset)

print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.000008,0.000006,3,1.000000,1.000000,1.000000,1.000000


{'eval_loss': 5.783319011243293e-06, 'eval_accuracy': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_f1': 1.0}


we should check for data leakage or duplicates.

In [25]:
test_results = trainer.evaluate(test_dataset)

print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.000008,0.000006,3,1.000000,1.000000,1.000000,1.000000


{'eval_loss': 5.783319011243293e-06, 'eval_accuracy': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_f1': 1.0}


In [26]:
print("Total reviews:", len(df))
print("Unique reviews:", df["text"].nunique())
print("Duplicate reviews:", df["text"].duplicated().sum())

Total reviews: 38432
Unique reviews: 36022
Duplicate reviews: 2410


In [27]:
train_texts = set(train_df["text"])
test_texts = set(test_df["text"])

overlap = train_texts.intersection(test_texts)

print("Train/Test identical reviews:", len(overlap))

Train/Test identical reviews: 299


In [28]:
print("Training:")
print(train_df["label"].value_counts())

print("\nValidation:")
print(val_df["label"].value_counts())

print("\nTesting:")
print(test_df["label"].value_counts())

Training:
label
2    26902
Name: count, dtype: int64

Validation:
label
2    5765
Name: count, dtype: int64

Testing:
label
2    5765
Name: count, dtype: int64


In [29]:
print(
    df["label"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

label
2    100.0
Name: proportion, dtype: float64


In [30]:
for i in range(20):
    print("LABEL:", df.iloc[i]["label"])
    print(df.iloc[i]["text"])
    print("-" * 80)

LABEL: 2
10/10 quality and feel really heavy and fit comfortably
--------------------------------------------------------------------------------
LABEL: 2
Yesterday i received my sunglasses parcel, it’ S really premium and one of the top quality acetates on alibaba, there are 1000’ S of suppliers here on alibaba but i always bought from qmoon sunglasses (this supplier). Because qmoon quality can’ T be compared with others. If alibaba have an option of 10+ star rating i would have given that for the quality of this acetates. Thanks to my qmoon family (especially damon, john li, candy, jenny, sunshine, hellen etc). This all the suppliers works as a team to receive you and they showcase the products with images and videos in order you to finalize the product. Thanks qmoon once again for your products and services. I am fully satisfied with your products.|...Show all
--------------------------------------------------------------------------------
LABEL: 2
The product is nice and shipping i

In [34]:
from sklearn.metrics import accuracy_score

majority_label = train_df["label"].mode()[0]

baseline_predictions = [majority_label] * len(test_df)

baseline_accuracy = accuracy_score(
    test_df["label"],
    baseline_predictions
)

print("Majority baseline accuracy:", baseline_accuracy)

Majority baseline accuracy: 1.0


In [35]:
print("===== DATASET =====")
print("Total:", len(df))
print("Unique:", df["text"].nunique())
print("Duplicates:", df["text"].duplicated().sum())

print("\n===== TRAIN/TEST OVERLAP =====")

train_texts = set(train_df["text"])
test_texts = set(test_df["text"])

overlap = train_texts.intersection(test_texts)

print("Identical train/test reviews:", len(overlap))

print("\n===== LABEL DISTRIBUTION =====")

print(df["label"].value_counts().sort_index())

print("\n===== TEST PERFORMANCE =====")

test_results = trainer.evaluate(test_dataset)

print(test_results)

===== DATASET =====
Total: 38432
Unique: 36022
Duplicates: 2410

===== TRAIN/TEST OVERLAP =====
Identical train/test reviews: 299

===== LABEL DISTRIBUTION =====
label
2    38432
Name: count, dtype: int64

===== TEST PERFORMANCE =====


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.000008,0.000006,3,1.000000,1.000000,1.000000,1.000000


{'eval_loss': 5.783319011243293e-06, 'eval_accuracy': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_f1': 1.0}


In [36]:
# 1. Test performance
test_results = trainer.evaluate(test_dataset)

print("===== TEST RESULTS =====")
print(test_results)


# 2. Duplicate check
print("\n===== DUPLICATE CHECK =====")
print("Total reviews:", len(df))
print("Unique reviews:", df["text"].nunique())
print("Duplicate reviews:", df["text"].duplicated().sum())


# 3. Train/Test overlap
print("\n===== TRAIN/TEST OVERLAP =====")

train_texts = set(train_df["text"])
test_texts = set(test_df["text"])

overlap = train_texts.intersection(test_texts)

print("Identical reviews in both:", len(overlap))


# 4. Label distribution
print("\n===== LABEL DISTRIBUTION =====")
print(df["label"].value_counts().sort_index())

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.000008,0.000006,3,1.000000,1.000000,1.000000,1.000000


===== TEST RESULTS =====
{'eval_loss': 5.783319011243293e-06, 'eval_accuracy': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_f1': 1.0}

===== DUPLICATE CHECK =====
Total reviews: 38432
Unique reviews: 36022
Duplicate reviews: 2410

===== TRAIN/TEST OVERLAP =====
Identical reviews in both: 299

===== LABEL DISTRIBUTION =====
label
2    38432
Name: count, dtype: int64


In [39]:
import pandas as pd

df_check = pd.read_csv("/content/all_product_review_summaries.csv")

print(df_check.columns.tolist())

['product_id', 'product_title', 'product_category_leaf_id', 'product_category_top_id', 'product_category_breadcrumb', 'product_brand', 'product_model_number', 'product_price_currency', 'product_price_min', 'product_price_max', 'product_moq', 'product_unit', 'product_sales_volume_text', 'product_sku_ids', 'product_default_sku_id', 'seller_company_id', 'seller_company_name', 'seller_business_type', 'seller_register_country', 'seller_years_on_platform', 'seller_average_rating', 'seller_total_reviewed_orders', 'seller_trade_assurance_enabled', 'seller_on_time_delivery_rate', 'review_source', 'review_id', 'reviewer_name', 'reviewer_country', 'review_rating', 'review_title', 'review_text', 'review_date', 'review_verified_purchase', 'review_repeat_buyer', 'review_helpful_votes', 'review_total_votes', 'review_price_at_review', 'review_currency', 'review_variant', 'review_images_count', 'data_source', 'scraped_at']


In [40]:
print(df_check["review_rating"].value_counts(dropna=False))

review_rating
5.0    55053
NaN        3
Name: count, dtype: int64


In [41]:
print(df_check.columns.tolist())

['product_id', 'product_title', 'product_category_leaf_id', 'product_category_top_id', 'product_category_breadcrumb', 'product_brand', 'product_model_number', 'product_price_currency', 'product_price_min', 'product_price_max', 'product_moq', 'product_unit', 'product_sales_volume_text', 'product_sku_ids', 'product_default_sku_id', 'seller_company_id', 'seller_company_name', 'seller_business_type', 'seller_register_country', 'seller_years_on_platform', 'seller_average_rating', 'seller_total_reviewed_orders', 'seller_trade_assurance_enabled', 'seller_on_time_delivery_rate', 'review_source', 'review_id', 'reviewer_name', 'reviewer_country', 'review_rating', 'review_title', 'review_text', 'review_date', 'review_verified_purchase', 'review_repeat_buyer', 'review_helpful_votes', 'review_total_votes', 'review_price_at_review', 'review_currency', 'review_variant', 'review_images_count', 'data_source', 'scraped_at']


In [42]:
for col in df_check.columns:
    print("\n", col)
    print(df_check[col].dropna().head(3).tolist())


 product_id
[1600459219006, 1600459219006, 1600459219006]

 product_title
['2026 High Quality Custom Logo Polarized Eyewear Retro Luxury Square Thick Acetate Shades Sunglasses for Men and Women', '2026 High Quality Custom Logo Polarized Eyewear Retro Luxury Square Thick Acetate Shades Sunglasses for Men and Women', '2026 High Quality Custom Logo Polarized Eyewear Retro Luxury Square Thick Acetate Shades Sunglasses for Men and Women']

 product_category_leaf_id
[33902, 33902, 33902]

 product_category_top_id
[36, 36, 36]

 product_category_breadcrumb
['Jewelry, Eyewear, Watches & Accessories > Eyewear > Sunglasses', 'Jewelry, Eyewear, Watches & Accessories > Eyewear > Sunglasses', 'Jewelry, Eyewear, Watches & Accessories > Eyewear > Sunglasses']

 product_brand
['QMOON', 'QMOON', 'QMOON']

 product_model_number
['QMZZ204', 'QMZZ204', 'QMZZ204']

 product_price_currency
['BDT', 'BDT', 'BDT']

 product_price_min
[912.66, 912.66, 912.66]

 product_price_max
[1058.19, 1058.19, 1058.19]

 p

In [43]:
print(df_check.columns.tolist())

['product_id', 'product_title', 'product_category_leaf_id', 'product_category_top_id', 'product_category_breadcrumb', 'product_brand', 'product_model_number', 'product_price_currency', 'product_price_min', 'product_price_max', 'product_moq', 'product_unit', 'product_sales_volume_text', 'product_sku_ids', 'product_default_sku_id', 'seller_company_id', 'seller_company_name', 'seller_business_type', 'seller_register_country', 'seller_years_on_platform', 'seller_average_rating', 'seller_total_reviewed_orders', 'seller_trade_assurance_enabled', 'seller_on_time_delivery_rate', 'review_source', 'review_id', 'reviewer_name', 'reviewer_country', 'review_rating', 'review_title', 'review_text', 'review_date', 'review_verified_purchase', 'review_repeat_buyer', 'review_helpful_votes', 'review_total_votes', 'review_price_at_review', 'review_currency', 'review_variant', 'review_images_count', 'data_source', 'scraped_at']
